# TC 5033
## Deep Learning
## Convolutional Neural Networks
#### Activity 2b: Building a CNN for CIFAR10 dataset with PyTorch

**Team Members:**
1. Joel Arturo Becerril Balderas - $A01797427$
2. Angel Eduardo Urueta Puello - $A01796724$
3. Marco Antonio Chávez García - $A01797547$
4. Efraín Paredes Balgañón - $A01351304$

In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
import torch.utils.data.sampler as sampler
import torchvision.datasets as datasets
import torchvision.transforms as T
import matplotlib.pyplot as plt
import warnings


# Seleccion de Hardware
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')
print(f'Entrenando en: {device}')

Entrenando en: cpu


### 1. Data Loading & Preprocessing

In [2]:
DATA_PATH = r'./data'
NUM_TRAIN = 50000
NUM_VAL = 5000
NUM_TEST = 5000
MINIBATCH_SIZE = 64

transform_cifar = T.Compose([
    T.ToTensor(),
    T.Normalize([0.491, 0.482, 0.447], [0.247, 0.243, 0.261])
])

# Train dataset
cifar10_train = datasets.CIFAR10(DATA_PATH, train=True, download=True, transform=transform_cifar)
train_loader = DataLoader(cifar10_train, batch_size=MINIBATCH_SIZE, sampler=sampler.SubsetRandomSampler(range(NUM_TRAIN)))

# Validation set
cifar10_val = datasets.CIFAR10(DATA_PATH, train=False, download=True, transform=transform_cifar)
val_loader = DataLoader(cifar10_val, batch_size=MINIBATCH_SIZE, sampler=sampler.SubsetRandomSampler(range(NUM_VAL)))

# Test set
cifar10_test = datasets.CIFAR10(DATA_PATH, train=False, download=True, transform=transform_cifar)
test_loader = DataLoader(cifar10_test, batch_size=MINIBATCH_SIZE, sampler=sampler.SubsetRandomSampler(range(NUM_VAL, len(cifar10_test))))

100.0%
c:\Users\baldj\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


### 2. Evaluation Metric (Accuracy)
This function evaluates the model on the provided dataset. We disable gradients using `torch.no_grad()` to save memory and set the model to `eval()` mode.

In [3]:
def accuracy(model, loader):
    num_correct = 0
    num_total = 0
    model.eval() # Set model to evaluation mode
    model = model.to(device=device)
    
    with torch.no_grad(): # Disable gradients for evaluation
        for x, y in loader:
            x = x.to(device=device, dtype=torch.float32)
            y = y.to(device=device, dtype=torch.long)
            
            scores = model(x)
            _, preds = scores.max(dim=1)
            
            num_correct += (preds == y).sum()
            num_total += preds.size(0)
            
        return float(num_correct)/num_total

### 3. Training Loop
We compute the cost using `F.cross_entropy` directly during each iteration, mirroring the fundamental tutorials for clear functional step-by-step optimization.

In [4]:
def train(model, optimiser, epochs=100):
    model = model.to(device=device)
    
    for epoch in range(epochs):
        for i, (x, y) in enumerate(train_loader):
            model.train() # Set to training mode
            
            x = x.to(device=device, dtype=torch.float32)
            y = y.to(device=device, dtype=torch.long)
            
            scores = model(x)
            cost = F.cross_entropy(input=scores, target=y)
            
            optimiser.zero_grad()
            cost.backward()
            optimiser.step()
            
        acc = accuracy(model, val_loader)
        print(f'Epoch: {epoch+1}, Costo: {cost.item():.4f}, Val Acc: {acc:.4f}')

### 4. Baseline Model (Fully Connected)
Flattening a 3D image destroys spatial context. We build this baseline to demonstrate that it plateaus around 50% accuracy, highlighting the need for Convolutional layers.

In [5]:
input_size = 3 * 32 * 32 
hidden_size = 512
num_classes = 10

model1 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(in_features=input_size, out_features=hidden_size),
    nn.ReLU(),
    nn.Linear(in_features=hidden_size, out_features=num_classes)
)

optimiser1 = torch.optim.Adam(model1.parameters(), lr=1e-3)

print('--- Entrenando Modelo Lineal Base ---')
train(model1, optimiser1, epochs=10)

--- Entrenando Modelo Lineal Base ---
Epoch: 1, Costo: 1.5933, Val Acc: 0.4514
Epoch: 2, Costo: 1.1017, Val Acc: 0.4660
Epoch: 3, Costo: 1.3943, Val Acc: 0.4862
Epoch: 4, Costo: 1.4408, Val Acc: 0.4876
Epoch: 5, Costo: 1.6113, Val Acc: 0.4856
Epoch: 6, Costo: 1.5362, Val Acc: 0.4932
Epoch: 7, Costo: 1.1161, Val Acc: 0.4992
Epoch: 8, Costo: 0.6926, Val Acc: 0.5202
Epoch: 9, Costo: 1.1605, Val Acc: 0.5166
Epoch: 10, Costo: 1.0639, Val Acc: 0.5210


### 5. Convolutional Neural Network (CNN)
We construct modular convolutional blocks (`Conv2d -> BatchNorm -> ReLU -> MaxPool`) and chain them in an `nn.Sequential` container, completely aligned with the class examples.

In [6]:
def crear_bloque_conv(in_channels, out_channels, kernel_size=3, padding=1):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2)
    )

canales_finales = 64
alto_final = 8
ancho_final = 8
neuronas_planas = canales_finales * alto_final * ancho_final

modelCNN1 = nn.Sequential(
    crear_bloque_conv(in_channels=3, out_channels=32),
    crear_bloque_conv(in_channels=32, out_channels=64),
    
    nn.Flatten(),
    nn.Linear(in_features=neuronas_planas, out_features=256),
    nn.ReLU(),
    nn.Dropout(p=0.5),
    nn.Linear(in_features=256, out_features=10)
)

optimiserCNN = torch.optim.Adam(modelCNN1.parameters(), lr=1e-3, weight_decay=1e-4)

print('\n--- Entrenando Modelo Convolucional (CNN) ---')
train(modelCNN1, optimiserCNN, epochs=10)


--- Entrenando Modelo Convolucional (CNN) ---
Epoch: 1, Costo: 1.6717, Val Acc: 0.5818
Epoch: 2, Costo: 1.5769, Val Acc: 0.6374
Epoch: 3, Costo: 1.4073, Val Acc: 0.6756
Epoch: 4, Costo: 0.7972, Val Acc: 0.6880
Epoch: 5, Costo: 0.5947, Val Acc: 0.7158
Epoch: 6, Costo: 0.5496, Val Acc: 0.7062
Epoch: 7, Costo: 1.1806, Val Acc: 0.7222
Epoch: 8, Costo: 0.5800, Val Acc: 0.7296
Epoch: 9, Costo: 1.0329, Val Acc: 0.7398
Epoch: 10, Costo: 0.8588, Val Acc: 0.7352


### 6. Final Evaluation
Testing the ultimate accuracy on the unseen Test Set.

In [7]:
test_acc = accuracy(modelCNN1, test_loader)
print(f'\n🏆 Precisión Final en Datos de Prueba (Test Set): {test_acc:.2%}')


🏆 Precisión Final en Datos de Prueba (Test Set): 73.84%
